In [49]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import procrustes as scipy_procrustes

from sklearn.decomposition import PCA

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader
from miscope.analysis.library.trajectory import (
    compute_arc_length,
    compute_curvature_profile,
    compute_signed_loop_area,
    detect_self_intersection,
    fit_centroid_pca,
    normalize_per_group,
)

In [50]:
FAMILY_NAME = "modulo_addition_1layer"
family = load_family(FAMILY_NAME)

# Named reference models (prime, model_seed, data_seed, label)
REFERENCE_MODELS = [
    (113, 999, 598, "p113 canon"),
    (109, 485, 598, "p109 reference healthy"),
    (101, 999, 598, "p101 open-loop"),
    (89,  999, 598, "p89 smooth-diverge"),
    (59,  485, 598, "p59 overshooter"),
]

## 0. Data loading

`load_trajectory` loads the full `parameter_snapshot` artifact for a variant,
flattens W_in at each checkpoint, fits PCA, and returns the PC projections.
All shape analysis operates on the 2D PC2vPC3 slice of this trajectory.

In [51]:
def load_trajectory(variant, epoch_limit=None, n_components=3):
    """Load W_in trajectory and return PCA projections.

    Returns:
        epochs:      list[int]
        coords:      np.ndarray (n_epochs, n_components)  — PC coordinates
        var_ratio:   np.ndarray (n_components,)
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    epochs = sorted(loader.get_epochs("parameter_snapshot"))
    if epoch_limit is not None:
        epochs = [e for e in epochs if e <= epoch_limit]

    rows = []
    for epoch in epochs:
        snap = loader.load_epoch("parameter_snapshot", epoch)
        rows.append(snap["W_in"].flatten())

    X = np.stack(rows)                          # (n_epochs, d_params)
    pca = PCA(n_components=n_components, random_state=0)
    coords = pca.fit_transform(X)               # (n_epochs, n_components)
    var_ratio = pca.explained_variance_ratio_
    return epochs, coords, var_ratio, X


def load_group_centroids(variant, epoch_limit=None):
    """Load per-group W_in centroids at every checkpoint.

    Returns:
        epochs:       list[int]
        centroids:    np.ndarray (n_groups, n_epochs, d_embed)
        group_freqs:  np.ndarray (n_groups,)
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    group_data = loader.load_cross_epoch("neuron_group_pca")
    neuron_group_idx = group_data["neuron_group_idx"]
    group_freqs = group_data["group_freqs"]
    n_groups = len(group_freqs)
    n_neurons = len(neuron_group_idx)

    epochs = sorted(loader.get_epochs("parameter_snapshot"))
    if epoch_limit is not None:
        epochs = [e for e in epochs if e <= epoch_limit]

    snap0 = loader.load_epoch("parameter_snapshot", epochs[0])
    d_first, d_second = snap0["W_in"].shape
    neuron_axis = 1 if d_second == n_neurons else 0
    d_embed = d_first if neuron_axis == 1 else d_second

    all_W = []
    for epoch in epochs:
        snap = loader.load_epoch("parameter_snapshot", epoch)
        all_W.append(snap["W_in"])
    W_epochs = np.stack(all_W)  # (n_epochs, d_first, d_second)

    centroids = np.zeros((n_groups, len(epochs), d_embed))
    for g in range(n_groups):
        members = np.where(neuron_group_idx == g)[0]
        if len(members) == 0:
            continue
        if neuron_axis == 1:
            centroids[g] = W_epochs[:, :, members].mean(axis=2)
        else:
            centroids[g] = W_epochs[:, members, :].mean(axis=1)

    return epochs, centroids, group_freqs

## 1. The lemniscate — single-model visualization

PC2 vs PC3 is where the self-intersection is most visible.  
The node is the point of closest self-approach with arc-length separation ≥ 15% of total arc.

Compare the canon model (p113/s999/ds598) against the reference healthy (p109/s485/ds598)
to confirm the structure appears across variants before quantifying it.

In [52]:
def plot_lemniscate(variant, label, min_arc_sep=0.15):
    """PC2vPC3 trajectory with node marked and epoch colormap."""
    epochs, coords, var_ratio, _ = load_trajectory(variant)
    curve = coords[:, 1:3]  # PC2, PC3
    node_result = detect_self_intersection(curve, min_arc_sep_fraction=min_arc_sep)

    colors = list(range(len(epochs)))
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=curve[:, 0], y=curve[:, 1],
        mode="markers",
        marker=dict(color=colors, colorscale="Blues", size=4,
                    colorbar=dict(title="epoch idx")),
        name="trajectory",
    ))
    nx, ny = node_result["node_position"]
    i, j = node_result["idx_pair"]
    fig.add_trace(go.Scatter(
        x=[nx], y=[ny],
        mode="markers",
        marker=dict(color="red", size=12, symbol="x"),
        name=f"node (ep {epochs[i]}–{epochs[j]})",
    ))
    fig.update_layout(
        title=f"{label} — PC2 vs PC3  (var: PC2={var_ratio[1]:.2%}, PC3={var_ratio[2]:.2%})",
        xaxis_title="PC2", yaxis_title="PC3",
        width=700, height=550,
    )
    return fig, node_result


for prime, seed, dseed, label in REFERENCE_MODELS[:2]:
    v = family.get_variant(prime=prime, seed=seed, data_seed=dseed)
    fig, _ = plot_lemniscate(v, label)
    fig.show()

## 2. Shape metrics — one model

Apply all four metrics to the canon model:
1. **Node location** — epoch indices at closest self-approach, min_distance
2. **Endpoint-to-node distance** — how far does the final point land from the node?
3. **Signed loop area** — magnitude = loop size; sign = traversal direction
4. **Curvature profile** — κ(s) resampled onto normalized arc-length [0, 1]

In [53]:
def compute_shape_metrics(variant, min_arc_sep=0.15):
    """Full set of lemniscate shape metrics for one variant.

    Returns a dict of scalar metrics plus the curvature profile arrays.
    """
    epochs, coords, var_ratio, X = load_trajectory(variant)
    curve = coords[:, 1:3]  # PC2vPC3

    node = detect_self_intersection(curve, min_arc_sep_fraction=min_arc_sep)
    arc = node["arc_length"]
    total_arc = arc[-1]
    i_node, j_node = node["idx_pair"]

    n_epochs = len(epochs)
    quartile_start = int(n_epochs * 0.75)
    late_X = X[quartile_start:]
    late_epochs = epochs[quartile_start:]
    if len(late_X) > 1:
        diffs = np.diff(late_X, axis=0)
        displacements = np.linalg.norm(diffs, axis=1)
        epoch_gaps = np.diff(late_epochs)
        late_velocity = float(np.mean(displacements / np.maximum(epoch_gaps, 1)))
    else:
        late_velocity = 0.0

    endpoint_dist = float(np.linalg.norm(curve[-1] - node["node_position"]))
    arc_frac_node = float(arc[i_node] / total_arc) if total_arc > 0 else 0.0
    # Normalised min-distance: near zero = genuine crossing; large = loop not completed
    node_min_dist_norm = node["min_distance"] / total_arc if total_arc > 0 else float("inf")

    loop_area = compute_signed_loop_area(curve, node["idx_pair"])
    curv_profile = compute_curvature_profile(curve)

    return {
        "epochs": epochs,
        "curve": curve,
        "node_position": node["node_position"],
        "node_epoch_pair": (epochs[i_node], epochs[j_node]),
        "node_min_distance": node["min_distance"],
        "node_min_dist_norm": node_min_dist_norm,
        "endpoint_to_node": endpoint_dist,
        "loop_area": loop_area,
        "arc_frac_node": arc_frac_node,
        "arc": arc,
        "curvature": curv_profile,
        "total_curvature": float(np.trapezoid(np.abs(curv_profile["kappa"]), curv_profile["s_norm"])),
        "kappa_variance": float(np.var(curv_profile["kappa"])),
        "late_velocity": late_velocity,
        "var_ratio": var_ratio,
    }


prime, seed, dseed, label = REFERENCE_MODELS[0]  # canon
v_canon = family.get_variant(prime=prime, seed=seed, data_seed=dseed)
m = compute_shape_metrics(v_canon)

print(f"Node epoch pair:       {m['node_epoch_pair']}")
print(f"Node min distance:     {m['node_min_distance']:.4f}  (0 = clean intersection)")
print(f"Arc frac at node:      {m['arc_frac_node']:.3f}  (fraction of journey to node)")
print(f"Endpoint-to-node:      {m['endpoint_to_node']:.4f}")
print(f"Node min dist (norm):  {m['node_min_dist_norm']:.4f}  (near 0 = genuine crossing)")
print(f"Signed loop area:      {m['loop_area']:.4f}")
print(f"Total curvature:       {m['total_curvature']:.4f}  (integral of |κ|)")
print(f"Kappa variance:        {m['kappa_variance']:.4f}")
print(f"Late velocity:         {m['late_velocity']:.6f}  (mean ||Δθ||/epoch in final 25% of training)")

Node epoch pair:       (1000, 17000)
Node min distance:     0.1036  (0 = clean intersection)
Arc frac at node:      0.278  (fraction of journey to node)
Endpoint-to-node:      1.0922
Node min dist (norm):  0.0024  (near 0 = genuine crossing)
Signed loop area:      62.0531
Total curvature:       0.1454  (integral of |κ|)
Kappa variance:        0.0267
Late velocity:         0.000594  (mean ||Δθ||/epoch in final 25% of training)


In [54]:
# Curvature profile — κ(s) across normalized arc-length
curv = m["curvature"]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=curv["s_norm"], y=curv["kappa"],
    mode="lines", name="κ(s)",
))
fig.add_vline(
    x=m["arc_frac_node"],
    line_dash="dash", line_color="red",
    annotation_text="node", annotation_position="top right",
)
fig.update_layout(
    title="Curvature profile κ(s) — canon model",
    xaxis_title="normalized arc-length s",
    yaxis_title="curvature κ",
    width=750, height=400,
)
fig.show()

## 3. Cross-variant survey

Run the shape metrics on all 30 variants and collect into a summary table.
Key columns: node epoch pair, endpoint-to-node distance, signed loop area, arc fraction at node.

Hypothesis: healthy models cluster at low endpoint-to-node distance;
overshooters show large positive loop area; never-arrives show near-zero loop area.

In [55]:
import pandas as pd

records = []
for v in family.list_variants():
    try:
        m = compute_shape_metrics(v)
        variant_name = f"p{v.params["prime"]}/s{v.params["seed"]}/ds{v.params["data_seed"]}"
        records.append({
            "variant": f"{variant_name}",
            "prime": v.params["prime"],
            "model_seed": v.params["seed"],
            "data_seed": v.params["data_seed"],
            "node_epoch_enter": m["node_epoch_pair"][0],
            "node_epoch_exit": m["node_epoch_pair"][1],
            "node_min_dist": m["node_min_distance"],
            "arc_frac_node": m["arc_frac_node"],
            "node_min_dist_norm": m["node_min_dist_norm"],
            "endpoint_to_node": m["endpoint_to_node"],
            "loop_area": m["loop_area"],
            "total_curvature": m["total_curvature"],
            "kappa_variance": m["kappa_variance"],
            "late_velocity": m["late_velocity"],
        })
    except Exception as ex:
        print(f"  skipped {v}: {ex}")

df = pd.DataFrame(records).sort_values("variant")
df

,variant,prime,model_seed,data_seed,node_epoch_enter,node_epoch_exit,node_min_dist,arc_frac_node,node_min_dist_norm,endpoint_to_node,loop_area,total_curvature,kappa_variance,late_velocity
31,p101/s485/ds42,101,485,42,3000,30000,0.304864,0.365748,0.006998,0.440302,50.500404,0.154839,0.029722,0.000704
18,p101/s485/ds598,101,485,598,2000,21000,0.164676,0.290833,0.003833,0.433160,-63.070786,0.226726,0.168249,0.001941
22,p101/s485/ds999,101,485,999,1200,20500,0.309566,0.351433,0.007761,0.559713,44.651077,0.147890,0.017052,0.000599
21,p101/s999/ds42,101,999,42,1000,15000,0.357595,0.277370,0.008428,1.820710,-59.337032,0.153737,0.029091,0.000686
17,p101/s999/ds598,101,999,598,3500,25700,0.268406,0.383026,0.006213,2.100883,-42.170357,0.154790,0.022370,0.000500
5,p101/s999/ds999,101,999,999,1200,13000,0.093160,0.285613,0.002552,2.579533,39.116814,0.154392,0.026395,0.000787
24,p103/s485/ds42,103,485,42,900,20500,0.090934,0.330290,0.002199,0.314075,48.296165,0.140188,0.031076,0.000499
33,p103/s485/ds598,103,485,598,900,23000,0.240416,0.344418,0.005818,0.165883,-44.753380,0.145948,0.060355,0.000461
16,p103/s485/ds999,103,485,999,800,18000,0.415075,0.269154,0.009311,4.930863,-56.728939,0.160193,0.241562,0.002602
19,p103/s999/ds42,103,999,42,1000,15000,0.332018,0.261668,0.007996,2.987063,55.738811,0.162301,0.037257,0.000763


In [56]:
# Scatter: endpoint-to-node vs signed loop area, coloured by prime
fig = go.Figure()
for prime_val in sorted(df["prime"].unique()):
    sub = df[df["prime"] == prime_val]
    fig.add_trace(go.Scatter(
        x=sub["loop_area"],
        y=sub["endpoint_to_node"],
        mode="markers+text",
        text=sub["variant"].str.replace(f"p{prime_val}/", ""),
        textposition="top center",
        name=f"p={prime_val}",
        marker=dict(size=8),
    ))
fig.update_layout(
    title="Endpoint-to-node distance vs loop area — all 30 variants",
    xaxis_title="signed loop area",
    yaxis_title="endpoint-to-node distance",
    width=850, height=600,
)
fig.show()

In [57]:
# Curvature profile overlay — reference models only
fig = go.Figure()
for prime, seed, dseed, label in REFERENCE_MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=dseed)
    m = compute_shape_metrics(v)
    curv = m["curvature"]
    fig.add_trace(go.Scatter(
        x=curv["s_norm"], y=curv["kappa"],
        mode="lines", name=label,
    ))
fig.update_layout(
    title="Curvature profiles κ(s) — reference models (arc-length normalized)",
    xaxis_title="normalized arc-length s",
    yaxis_title="curvature κ",
    width=850, height=450,
)
fig.show()

## 4. Per-group centroid trajectory shape

Does each frequency group run the same program as the global trajectory?

Approach:
1. Fit shared PCA on all group centroids across all epochs → shared coordinate frame
2. Extract the PC2vPC3 curve for each group's centroid trajectory
3. Apply `detect_self_intersection` and `compute_curvature_profile` per group
4. Cross-group Procrustes: are the per-group arc-length-normalized curves the same shape?

If yes → "turtles all the way down": the canonical lemniscate shape recurs at every scale.

In [58]:
# --- Load canon model group centroids and project into shared PC space ---
prime, seed, dseed, label = REFERENCE_MODELS[0]
v_canon = family.get_variant(prime=prime, seed=seed, data_seed=dseed)

epochs, centroids, group_freqs = load_group_centroids(v_canon)
pca_out = fit_centroid_pca(centroids, n_components=3)
coords = pca_out["coords"]  # (n_groups, n_epochs, 3)

print(f"n_groups: {len(group_freqs)}, group_freqs: {group_freqs}")
print(f"explained variance: {pca_out['explained_variance_ratio'].round(3)}")

n_groups: 4, group_freqs: [ 8 32 37 54]
explained variance: [0.338 0.16  0.121]


In [59]:
# Per-group PC2vPC3 trajectories — overlay with node markers
fig = go.Figure()
colors_group = [f"hsl({int(360 * i / len(group_freqs))}, 70%, 50%)" for i in range(len(group_freqs))]

group_metrics = []
for g, freq in enumerate(group_freqs):
    curve_g = coords[g, :, 1:3]  # PC2vPC3 for this group
    node_g = detect_self_intersection(curve_g)
    area_g = compute_signed_loop_area(curve_g, node_g["idx_pair"])
    curv_g = compute_curvature_profile(curve_g)
    ep_dist_g = float(np.linalg.norm(curve_g[-1] - node_g["node_position"]))

    group_metrics.append({
        "freq": int(freq), "loop_area": area_g,
        "endpoint_to_node": ep_dist_g,
        "node_min_dist": node_g["min_distance"],
        "curvature": curv_g,
    })

    fig.add_trace(go.Scatter(
        x=curve_g[:, 0], y=curve_g[:, 1],
        mode="lines",
        line=dict(color=colors_group[g], width=1.5),
        name=f"freq {freq}",
    ))
    nx, ny = node_g["node_position"]
    fig.add_trace(go.Scatter(
        x=[nx], y=[ny],
        mode="markers",
        marker=dict(color=colors_group[g], size=8, symbol="x"),
        showlegend=False,
    ))

fig.update_layout(
    title=f"{label} — per-group centroid PC2vPC3 trajectories",
    xaxis_title="PC2", yaxis_title="PC3",
    width=800, height=600,
)
fig.show()

pd.DataFrame(group_metrics)

,freq,loop_area,endpoint_to_node,node_min_dist,curvature
0,8,-0.000062,0.082176,0.007852,"{'s_norm': [0.0, 0.010101010101010102, 0.02020..."
1,32,-0.000094,0.030288,0.003168,"{'s_norm': [0.0, 0.010101010101010102, 0.02020..."
2,37,-0.000119,0.031603,0.001659,"{'s_norm': [0.0, 0.010101010101010102, 0.02020..."
3,54,-0.000022,0.017680,0.000625,"{'s_norm': [0.0, 0.010101010101010102, 0.02020..."


In [60]:
# Curvature profile overlay — all groups on the same normalized arc-length axis
fig = go.Figure()
for g, gm in enumerate(group_metrics):
    curv = gm["curvature"]
    fig.add_trace(go.Scatter(
        x=curv["s_norm"], y=curv["kappa"],
        mode="lines",
        line=dict(color=colors_group[g], width=1.5),
        name=f"freq {gm['freq']}",
    ))
fig.update_layout(
    title=f"{label} — per-group curvature profiles (arc-length normalized)",
    xaxis_title="normalized arc-length s",
    yaxis_title="curvature κ",
    width=800, height=450,
)
fig.show()

In [61]:
# Cross-group Procrustes: are the per-group PC2vPC3 shapes the same?
# Normalize each group's trajectory to unit arc-length scale before comparing.
n_groups = len(group_freqs)
D = np.zeros((n_groups, n_groups))

# Resample each group's curve onto a fixed number of arc-length-uniform points
N_RESAMPLE = 200
resampled = []
for g in range(n_groups):
    curve_g = coords[g, :, 1:3]
    arc_g = compute_arc_length(curve_g)
    s_uniform = np.linspace(0, arc_g[-1], N_RESAMPLE)
    x_r = np.interp(s_uniform, arc_g, curve_g[:, 0])
    y_r = np.interp(s_uniform, arc_g, curve_g[:, 1])
    resampled.append(np.column_stack([x_r, y_r]))

for i in range(n_groups):
    for j in range(i + 1, n_groups):
        try:
            _, _, disp = scipy_procrustes(resampled[i], resampled[j])
        except Exception:
            disp = np.nan
        D[i, j] = D[j, i] = disp

fig = go.Figure(go.Heatmap(
    z=D,
    x=[f"f{int(f)}" for f in group_freqs],
    y=[f"f{int(f)}" for f in group_freqs],
    colorscale="Blues_r",
    zmin=0, zmax=0.5,
    colorbar=dict(title="Procrustes disparity"),
))
fig.update_layout(
    title=f"{label} — cross-group Procrustes disparity (PC2vPC3 shapes)",
    width=600, height=550,
)
fig.show()

upper = D[np.triu_indices(n_groups, k=1)]
print(f"Mean Procrustes disparity: {np.nanmean(upper):.4f}  (0=identical shape, 1=maximally different)")

Mean Procrustes disparity: 0.2602  (0=identical shape, 1=maximally different)


## 5. Epoch-truncated shape metrics — when does the lemniscate emerge?

For each model, compute `|loop_area|` and `node_min_dist_norm` for the first T epochs
of the trajectory in the fixed final-epoch PCA basis, for increasing T.

**Hypothesis:** loop area ≈ 0 through the plateau; rises sharply at second descent onset —
mirroring the R²_curvature rise seen in the intragroup_manifold timeseries.

Two panels:
1. Absolute epoch — raw timing per model
2. Epoch relative to `second_descent_onset_epoch` — aligns models to show whether
   shape emergence is consistently tied to that landmark

In [62]:
import json as _json


def load_timing(variant) -> dict:
    """Load key epoch markers from variant_summary.json."""
    summary_path = variant.variant_dir / 'variant_summary.json'
    summary = _json.loads(summary_path.read_text())
    return {
        'second_descent_onset': summary.get('second_descent_onset_epoch'),
        'grokking':             summary.get('test_loss_threshold_first_epoch'),
    }


def compute_truncated_loop_metrics(variant, min_points: int = 20) -> dict:
    """Loop area and node proximity at every epoch cutoff T.

    Uses the fixed final-epoch PCA basis. For each T from min_points to
    n_epochs, runs detect_self_intersection and compute_signed_loop_area
    on curve[:T] (PC2vPC3 slice).

    Returns:
        epochs:             (n_epochs,) full epoch array
        loop_area:          (n_epochs - min_points + 1,) signed loop area at each T
        node_min_dist_norm: (n_epochs - min_points + 1,) normalised node distance
        cutoff_epochs:      epochs corresponding to each T value
    """
    epochs, coords, _, _ = load_trajectory(variant)
    curve_full = coords[:, 1:3]  # PC2vPC3

    loop_areas, node_dists, cutoff_epochs = [], [], []
    for T in range(min_points, len(epochs) + 1):
        curve = curve_full[:T]
        node = detect_self_intersection(curve)
        total_arc = node['arc_length'][-1]
        area = compute_signed_loop_area(curve, node['idx_pair'])
        dist_norm = node['min_distance'] / total_arc if total_arc > 1e-12 else float('inf')
        loop_areas.append(area)
        node_dists.append(dist_norm)
        cutoff_epochs.append(epochs[T - 1])

    return {
        'epochs':             np.array(epochs),
        'loop_area':          np.array(loop_areas),
        'node_min_dist_norm': np.array(node_dists),
        'cutoff_epochs':      np.array(cutoff_epochs),
    }

In [63]:
# Run truncated metrics for all reference models
truncated_results = {}
for prime, seed, dseed, label in REFERENCE_MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=dseed)
    truncated_results[label] = {
        'metrics': compute_truncated_loop_metrics(v),
        'timing':  load_timing(v),
        'label':   label,
    }
    sd = truncated_results[label]['timing']['second_descent_onset']
    gk = truncated_results[label]['timing']['grokking']
    print(f"{label}  second_descent={sd}  grokking={gk}")

p113 canon  second_descent=9503  grokking=12448
p109 reference healthy  second_descent=3584  grokking=5243
p101 open-loop  second_descent=13043  grokking=-1
p89 smooth-diverge  second_descent=20699  grokking=24977
p59 overshooter  second_descent=15553  grokking=-1


In [64]:
# Panel 1 — absolute epoch
# Panel 2 — epoch relative to second_descent_onset
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        '|loop_area| vs absolute epoch',
        '|loop_area| vs epoch relative to second_descent_onset',
    ],
    vertical_spacing=0.12,
)

for idx, (label, res) in enumerate(truncated_results.items()):
    m   = res['metrics']
    t   = res['timing']
    col = colors[idx]
    ep  = m['cutoff_epochs']
    area = np.abs(m['loop_area'])

    # Panel 1 — absolute
    fig.add_trace(
        go.Scatter(x=ep, y=area, mode='lines', name=label,
                   line=dict(color=col, width=1.5), legendgroup=label),
        row=1, col=1,
    )
    # second_descent marker
    if t['second_descent_onset'] is not None:
        fig.add_vline(x=t['second_descent_onset'], line_dash='dash',
                      line_color=col, line_width=1, row=1, col=1)

    # Panel 2 — relative to second_descent_onset
    if t['second_descent_onset'] is not None:
        rel_ep = ep - t['second_descent_onset']
        fig.add_trace(
            go.Scatter(x=rel_ep, y=area, mode='lines', name=label,
                       line=dict(color=col, width=1.5),
                       legendgroup=label, showlegend=False),
            row=2, col=1,
        )

fig.add_vline(x=0, line_dash='dot', line_color='gray', line_width=1.5,
              annotation_text='second_descent_onset', row=2, col=1)

fig.update_xaxes(title_text='epoch', row=1, col=1)
fig.update_xaxes(title_text='epoch − second_descent_onset', row=2, col=1)
fig.update_yaxes(title_text='|loop_area|', row=1, col=1)
fig.update_yaxes(title_text='|loop_area|', row=2, col=1)
fig.update_layout(
    title='Lemniscate loop area emergence over training',
    height=750, width=900,
)
fig.show()

In [65]:
# Companion: node_min_dist_norm — how 'genuine' is the crossing at each T?
# Near 0 = real self-intersection; large = loop not yet closed.
# Cap display at 1.0 so early-epoch spikes don't dominate the y-axis.

fig2 = go.Figure()
for idx, (label, res) in enumerate(truncated_results.items()):
    m   = res['metrics']
    t   = res['timing']
    col = colors[idx]
    ep  = m['cutoff_epochs']
    dist = np.clip(m['node_min_dist_norm'], 0, 1.0)

    fig2.add_trace(go.Scatter(
        x=ep, y=dist, mode='lines', name=label,
        line=dict(color=col, width=1.5),
    ))
    if t['second_descent_onset'] is not None:
        fig2.add_vline(x=t['second_descent_onset'], line_dash='dash',
                       line_color=col, line_width=1)

fig2.update_layout(
    title='Node proximity (node_min_dist_norm, capped at 1) — dashed = second_descent_onset',
    xaxis_title='epoch',
    yaxis_title='node_min_dist_norm (lower = cleaner crossing)',
    height=450, width=900,
)
fig2.show()

## 6. Per-group centroid trajectory range — min, max, final per PC

For each frequency group's centroid trajectory (projected into the shared centroid PCA
basis fitted across all groups and epochs), report the min, max, and final value of
each of PC1, PC2, PC3.

The basis is fitted independently per model, so values are comparable across groups
within a model but not directly across models.

In [66]:
records = []
for prime, seed, dseed, label in REFERENCE_MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=dseed)
    _, centroids, group_freqs = load_group_centroids(v)
    pca_out = fit_centroid_pca(centroids, n_components=3)
    coords = pca_out['coords']  # (n_groups, n_epochs, 3)

    for g, freq in enumerate(group_freqs):
        traj = coords[g]  # (n_epochs, 3)
        row = {
            'model':      label,
            'freq':       int(freq) + 1,  # 1-indexed
        }
        for pc_idx, pc_name in enumerate(['PC1', 'PC2', 'PC3']):
            series = traj[:, pc_idx]
            row[f'{pc_name}_min']   = float(series.min())
            row[f'{pc_name}_max']   = float(series.max())
            row[f'{pc_name}_final'] = float(series[-1])
        records.append(row)

df_range = pd.DataFrame(records)

# Round for readability
float_cols = [c for c in df_range.columns if c not in ('model', 'freq')]
df_range[float_cols] = df_range[float_cols].round(4)

df_range

,model,freq,PC1_min,PC1_max,PC1_final,PC2_min,PC2_max,PC2_final,PC3_min,PC3_max,PC3_final
0,p113 canon,9,-0.0483,0.0236,-0.0483,-0.0134,0.0836,-0.0134,-0.0205,-0.0044,-0.0126
1,p113 canon,33,-0.0009,0.0699,0.0004,-0.0002,0.0326,0.0004,0.0021,0.0528,0.0040
2,p113 canon,38,-0.0387,0.0918,-0.0387,-0.0386,-0.0033,-0.0073,-0.0565,-0.0001,-0.0001
3,p113 canon,55,-0.0782,0.0410,-0.0782,-0.0412,-0.0061,-0.0168,0.0094,0.0441,0.0163
4,p109 reference healthy,4,-0.0486,0.0835,0.0775,-0.0427,0.0189,-0.0427,-0.0022,0.0137,0.0116
5,p109 reference healthy,14,-0.0803,0.0478,0.0478,-0.0718,0.0540,0.0517,-0.0688,-0.0021,-0.0409
6,p109 reference healthy,27,-0.1056,-0.0099,-0.0099,0.0015,0.0213,0.0015,0.0137,0.0445,0.0445
7,p101 open-loop,35,-0.0249,0.0508,0.0499,0.0072,0.1064,0.0072,-0.0257,0.0542,-0.0257
8,p101 open-loop,41,-0.0211,0.0414,0.0414,-0.0466,-0.0055,-0.0379,0.0064,0.0655,0.0096
9,p101 open-loop,43,-0.0829,0.0215,0.0215,-0.0393,-0.0032,-0.0032,-0.0286,0.0215,-0.0286
